# HAB Predictor — Weighted Random Forest Baseline

Predict whether a given pier-week is a Harmful Algal Bloom (`pda > 0.1`).

- **Data:** `merged_hab_oisst_features.csv` (Jin's branch — already has lag features and Si:N ratio)
- **Target:** `isHarmful` (1 if `pda > 0.1`, else 0)
- **Class balance:** 195 harmful / 2883 quiet (~6.3% positive)
- **Primary metric:** **Recall** on the harmful class (per Katie — false alarms are cheap, missed blooms are not)
- **Imbalance handling:** `class_weight={0: 1, 1: 50}`
- **Split:** time-based (random splits leak the future)

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
)

## Load data

In [ ]:
df = pd.read_csv('merged_hab_oisst_features.csv', parse_dates=['week_start'])
df = df.sort_values('week_start').reset_index(drop=True)
print('shape:', df.shape)
print('date range:', df['week_start'].min(), '->', df['week_start'].max())
print('stations:', df['station'].nunique())
df.head()

## Class balance

In [ ]:
counts = df['isHarmful'].value_counts().sort_index()
print(counts)
print('positive rate:', counts.get(1, 0) / counts.sum())

In [ ]:
per_station = (
    df.groupby('station')
      .agg(weeks=('isHarmful', 'size'), harmful_weeks=('isHarmful', 'sum'))
      .assign(harmful_rate=lambda d: d['harmful_weeks'] / d['weeks'])
      .sort_values('harmful_rate', ascending=False)
)
per_station

## Features & target

In [ ]:
FEATURES = [
    'temp', 'silicate', 'nitrate', 'avg_chloro',
    'temp_lag1', 'temp_lag2',
    'silicate_lag1', 'silicate_lag2',
    'nitrate_lag1', 'nitrate_lag2',
    'avg_chloro_lag1', 'avg_chloro_lag2',
    'silicate_nitrate_ratio',
    'sst_roll_14d', 'anom_roll_14d', 'sst_roc_3d',
    'warm_degree_days_14d', 'above_avg',
    'latitude', 'longitude', 'month',
]
TARGET = 'isHarmful'

model_df = df.dropna(subset=FEATURES + [TARGET]).copy()
print('rows after dropping NaN:', len(model_df), 'of', len(df))
print('positive rate after drop:', model_df[TARGET].mean())

## Correlation analysis (deliverable #3)

Per Katie: *find correlations between some of the predictor columns and the pda column.*

Pearson catches linear relationships; Spearman is more honest here because `pda` is heavily zero-inflated with a long right tail. Sorting by `|spearman|`.

In [ ]:
pearson = model_df[FEATURES + ['pda']].corr(method='pearson')['pda'].drop('pda')
spearman = model_df[FEATURES + ['pda']].corr(method='spearman')['pda'].drop('pda')

corr_table = pd.DataFrame({'pearson': pearson, 'spearman': spearman})
corr_table = corr_table.reindex(corr_table['spearman'].abs().sort_values().index)

fig, ax = plt.subplots(figsize=(9, 7))
corr_table.plot.barh(ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with pda (sorted by |Spearman|)')
ax.set_xlabel('correlation')
plt.tight_layout()
plt.show()

corr_table.iloc[::-1]

In [ ]:
corr_matrix = model_df[FEATURES + ['pda']].corr(method='spearman')
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(corr_matrix.values, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticklabels(corr_matrix.columns)
fig.colorbar(im, ax=ax, fraction=0.04)
ax.set_title('Spearman correlation matrix (predictors + pda)')
plt.tight_layout()
plt.show()

## Time-based train/test split

Train on everything before the cutoff, test on everything after. A random split would leak future weeks into training.

In [ ]:
cutoff = model_df['week_start'].quantile(0.8)
train = model_df[model_df['week_start'] < cutoff]
test = model_df[model_df['week_start'] >= cutoff]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print('cutoff:', cutoff)
print('train size:', len(train), '| harmful:', int(y_train.sum()))
print('test size: ', len(test), '| harmful:', int(y_test.sum()))

## Weighted Random Forest — `class_weight={0: 1, 1: 50}`

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight={0: 1, 1: 50},
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['quiet', 'harmful'], digits=3))
print('confusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, y_pred))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 3))

## Compare with `class_weight='balanced'`

In [ ]:
rf_bal = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_bal.fit(X_train, y_train)
y_pred_bal = rf_bal.predict(X_test)
print(classification_report(y_test, y_pred_bal, target_names=['quiet', 'harmful'], digits=3))
print(confusion_matrix(y_test, y_pred_bal))

## Dummy baseline (always predict quiet)

Sanity check — the 97% accuracy trap Katie warned about.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print(classification_report(y_test, dummy.predict(X_test), target_names=['quiet', 'harmful'], digits=3, zero_division=0))

## Feature importances

In [ ]:
importances = (
    pd.Series(rf.feature_importances_, index=FEATURES)
      .sort_values(ascending=True)
)
fig, ax = plt.subplots(figsize=(8, 6))
importances.plot.barh(ax=ax)
ax.set_title('Random Forest feature importances (weighted 1:50)')
ax.set_xlabel('importance')
plt.tight_layout()
plt.show()

## Precision-Recall curve

Lets us pick a probability threshold that trades precision for more recall.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision)
ax.set_xlabel('Recall (harmful)')
ax.set_ylabel('Precision (harmful)')
ax.set_title('Precision-Recall curve — weighted RF')
ax.grid(True, alpha=0.3)
plt.show()